# 03 — Post-training evaluation

Re-run the same 50 seeds × 3 scenarios that notebook 01 used, but with the
GRPO-trained Qwen2.5-1.5B-Instruct + LoRA adapter from notebook 02. Plot
before/after curves on identical axes and emit the headline summary table
for the README.

Inputs (placed in `_artifacts/` by the prior notebooks):
* `baseline_metrics.json`
* `qwen_cybersec_lora/`

Outputs:
* `_artifacts/post_train_metrics.json`
* `_artifacts/before_after_curves.png`
* `_artifacts/summary_table.md`

## 0. Install

In [ ]:
%%capture
!pip install -q -e ../cybersec
!pip install -q --upgrade transformers peft accelerate bitsandbytes
!pip install -q matplotlib pandas

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from cybersec import (
    ActionType,
    CybersecAction,
    CybersecEnvironment,
    list_scenarios,
)
from cybersec.baselines import (
    HeuristicPolicy,
    RandomPolicy,
    aggregate_results,
    run_episode,
)

ARTIFACTS = Path("_artifacts")
ADAPTER_DIR = ARTIFACTS / "qwen_cybersec_lora"
BASELINE_METRICS = ARTIFACTS / "baseline_metrics.json"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

N_EPISODES = 50
SCENARIOS = list_scenarios()
SEEDS = list(range(N_EPISODES))

## 1. Load Qwen + trained adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

if ADAPTER_DIR.exists():
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
    print("loaded LoRA adapter from", ADAPTER_DIR)
else:
    model = base_model
    print("WARNING: no adapter found at", ADAPTER_DIR, "-- evaluating base model")
model.eval()

## 2. LLM defender policy

Identical to the prompt format used in notebook 01 / 02.

In [ ]:
SYSTEM_PROMPT = (
    "You are an SRE-grade cyber-defender driving an OpenEnv environment.\n"
    "Reply with exactly one JSON object on one line of the form\n"
    '{"action_type": "...", "target": "..."}.\n'
    "action_type must be one of MONITOR, INVESTIGATE, ISOLATE_ASSET, REVOKE_IDENTITY, BLOCK_EGRESS, PATCH_ASSET.\n"
    "target must be omitted (or null) for MONITOR; otherwise it must come from valid_targets."
)

def render_observation(obs) -> str:
    lines = [
        f"tick={obs.tick}/{obs.horizon}  scenario={obs.scenario_id}  attacker={obs.attacker_personality.value}",
        f"isolated_assets={obs.isolated_assets}",
        f"revoked_identities={obs.revoked_identities}",
        f"blocked_egress={obs.blocked_egress_assets}",
        f"patched={obs.patched_assets}",
        f"confirmed_compromised={obs.confirmed_compromised}",
        f"valid_targets={obs.valid_targets}",
        "recent_alerts:",
    ]
    for a in obs.alerts[-6:]:
        lines.append(f"  t={a.tick} {a.signal.value} sev={a.severity} asset={a.asset} id={a.identity} :: {a.description}")
    lines.append("recent_forensics:")
    for f in obs.forensics[-4:]:
        lines.append(f"  t={f.tick} {f.target_kind}={f.target} compromised={f.is_compromised} conf={f.confidence}")
    return "\n".join(lines)

def _parse_first_json_object(text: str):
    text = text.strip()
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start : i + 1])
                except json.JSONDecodeError:
                    return None
    return None

@torch.inference_mode()
def llm_act(obs):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": render_observation(obs)},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=48,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    payload = _parse_first_json_object(completion)
    if not payload:
        return CybersecAction(action_type=ActionType.MONITOR)
    try:
        return CybersecAction(**payload)
    except Exception:
        return CybersecAction(action_type=ActionType.MONITOR)

class LLMPolicy:
    name = "qwen-1.5b-grpo"
    def reset(self):
        pass
    def act(self, obs):
        return llm_act(obs)

## 3. Run trained-LLM, Random, Heuristic on the same seeds

Random and Heuristic re-run here so the comparison is on a single matplotlib
axis without re-loading them from the baseline JSON.

In [ ]:
env = CybersecEnvironment()
results = {}

for scenario_id in SCENARIOS:
    rand = [run_episode(env, RandomPolicy(seed=s), seed=s, scenario_id=scenario_id) for s in SEEDS]
    heur = [run_episode(env, HeuristicPolicy(),    seed=s, scenario_id=scenario_id) for s in SEEDS]
    llm  = [run_episode(env, LLMPolicy(),          seed=s, scenario_id=scenario_id) for s in SEEDS]
    results[scenario_id] = {"random": rand, "heuristic": heur, "llm": llm}
    print(scenario_id)
    print("  random   :", aggregate_results(rand))
    print("  heuristic:", aggregate_results(heur))
    print("  llm      :", aggregate_results(llm))

## 4. Before/after reward curves on identical axes

In [ ]:
def _padded_cumulative(curves, target_len):
    out = np.zeros((len(curves), target_len), dtype=float)
    for i, c in enumerate(curves):
        out[i, : len(c)] = c
        if len(c) < target_len:
            out[i, len(c):] = c[-1] if c else 0.0
    return np.cumsum(out, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, scenario_id in zip(axes, SCENARIOS):
    cell = results[scenario_id]
    horizon = max(
        max(len(r.reward_curve) for r in cell['random']),
        max(len(r.reward_curve) for r in cell['heuristic']),
        max(len(r.reward_curve) for r in cell['llm']),
    )
    palette = [("random", "tab:gray"), ("heuristic", "tab:blue"), ("llm", "tab:red")]
    for label, color in palette:
        cumr = _padded_cumulative([r.reward_curve for r in cell[label]], horizon)
        mean = cumr.mean(axis=0)
        std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    ax.set_title(scenario_id, fontsize=10)
    ax.set_xlabel("tick")
    ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward")
axes[0].legend()
fig.suptitle(f"Cybersec OpenEnv — pre/post GRPO training, n={N_EPISODES}/scenario")
fig.tight_layout()
fig.savefig(ARTIFACTS / "before_after_curves.png", dpi=140)
plt.show()

## 5. Summary table

In [ ]:
rows = []
for scenario_id, cell in results.items():
    for policy in ("random", "heuristic", "llm"):
        runs = cell[policy]
        agg = aggregate_results(runs)
        returns = [r.cumulative_reward for r in runs]
        rows.append({
            "scenario": scenario_id,
            "policy": policy,
            "mean_return": agg["mean_return"],
            "std_return": round(float(np.std(returns)), 3) if returns else 0.0,
            "mean_stages": agg["mean_stages_succeeded"],
            "exfil_rate": agg["exfil_rate"],
            "invalid_rate": agg["mean_invalid_actions"],
        })
df = pd.DataFrame(rows)
df

In [ ]:
post = {
    scenario_id: {p: aggregate_results(cell[p]) for p in ("random", "heuristic", "llm")}
    for scenario_id, cell in results.items()
}
post["_meta"] = {"n_episodes": N_EPISODES, "seeds": SEEDS, "model": MODEL_NAME, "adapter": str(ADAPTER_DIR)}
(ARTIFACTS / "post_train_metrics.json").write_text(json.dumps(post, indent=2))
(ARTIFACTS / "summary_table.md").write_text(df.to_markdown(index=False))
print("wrote", ARTIFACTS / "post_train_metrics.json")
print("wrote", ARTIFACTS / "summary_table.md")

## 6. Headline delta

Headline number for the README: trained LLM mean return vs. its zero-shot
baseline if notebook 01's `_llm_zeroshot` block was run, otherwise vs. the
heuristic. Falls back gracefully if the prior file is missing.

In [ ]:
if BASELINE_METRICS.exists():
    base = json.loads(BASELINE_METRICS.read_text())
    print("=== Headline deltas vs. notebook-01 baselines ===")
    for scenario_id in SCENARIOS:
        if scenario_id not in base:
            continue
        b = base[scenario_id]
        post_llm = aggregate_results(results[scenario_id]["llm"])
        anchor = base.get("_llm_zeroshot", {}).get(scenario_id, b.get("heuristic"))
        anchor_label = "llm-zeroshot" if base.get("_llm_zeroshot") else "heuristic"
        delta = post_llm["mean_return"] - anchor["mean_return"]
        print(f"{scenario_id:<32s}  llm-trained={post_llm['mean_return']:.3f}  vs {anchor_label}={anchor['mean_return']:.3f}  delta={delta:+.3f}")
else:
    print("baseline_metrics.json not found - run notebook 01 first to enable the delta print.")